# ForwardGNN — GAT & SAGE across PubMed, GitHub, Amazon-Photo

Extends the GCN/PubMed reproduction to the full architecture x dataset x method grid, **with per-run peak GPU memory recorded**.

**Grid:** 3 datasets x 2 architectures x 4 methods = 24 configs x 5 splits = 120 runs, all at `num_layers=4`.

**Prerequisite:** run cells 1-5 of the first notebook (Drive mount, repo clone, Python 3.10 env). This notebook assumes `VENV_PY`, `PROJECT_DIR` and `SRC` already exist. If the session is fresh, re-run those cells first.

## 0. Re-establish paths (if this is a fresh session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/forwardgnn-experiments'
VENV_PY     = '/content/fgnn-env/bin/python'
REPO        = '/content/forwardgnn'
SRC         = f'{REPO}/src'

import os, subprocess, shutil
assert os.path.exists(VENV_PY), "Env missing - re-run cells 1-5 of notebook 1."
if not os.path.exists(REPO):
    subprocess.run(['git','clone','-q','https://github.com/facebookresearch/forwardgnn.git',REPO], check=True)
for sub in ['data','datasplits','results']:
    p = f'{REPO}/{sub}'
    if os.path.islink(p): os.unlink(p)
    elif os.path.isdir(p): shutil.rmtree(p)
    os.symlink(f'{PROJECT_DIR}/{sub}', p)
print(subprocess.run([VENV_PY,'-c','import torch;print(torch.__version__, torch.cuda.get_device_name(0))'],
                     capture_output=True, text=True).stdout)

## 1. Install the memory-tracking launcher

The repo records accuracy, time and epochs — nothing about memory. This wrapper patches `ResultManager.save_run_result` so every result JSON also carries peak GPU memory, then resets the counters so each run is measured independently.

Two numbers get recorded:

- `gpu_mem_alloc_peak_mb` — peak **tensor** memory. This is the number to report: it reflects what the method actually needs, and it is where backprop and forward learning genuinely differ, because backprop must retain activations for the backward pass while forward learning does not.
- `gpu_mem_reserved_peak_mb` — peak **allocator pool**, closer to what `nvidia-smi` shows. Noisier, since PyTorch's caching allocator over-reserves.

Nothing in the repo is modified; this is a launcher placed alongside the training scripts.

In [ ]:
WRAPPER = r"""
# Copyright-free wrapper. Place at forwardgnn/src/run_with_mem.py
"""Launcher that records peak GPU memory per run into the repo's result JSONs.

Usage (from forwardgnn/src):
    python run_with_mem.py train_backprop.py --model GNN-GAT --dataset GitHub ...

Works by patching ResultManager.save_run_result so every result file gains
    gpu_mem_alloc_peak_mb      peak tensor memory  (torch.cuda.max_memory_allocated)
    gpu_mem_reserved_peak_mb   peak allocator pool (torch.cuda.max_memory_reserved)
and resetting the peak counters after each run so values are per-run, not cumulative.
"""
import os
# must be set before cuBLAS initialises; both repo scripts set this at import time
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
import runpy

if len(sys.argv) < 2:
    raise SystemExit("usage: run_with_mem.py <train_script.py> [args...]")

script = sys.argv[1]
sys.path.insert(0, os.path.dirname(os.path.abspath(script)) or ".")

import torch
from utils.train_utils import ResultManager

_orig_save = ResultManager.save_run_result


def _save_with_mem(self, run_i, perf_dict, num_layers=None):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        perf_dict = dict(perf_dict)
        perf_dict["gpu_mem_alloc_peak_mb"] = torch.cuda.max_memory_allocated() / 1024 ** 2
        perf_dict["gpu_mem_reserved_peak_mb"] = torch.cuda.max_memory_reserved() / 1024 ** 2
        perf_dict["gpu_name"] = torch.cuda.get_device_name(0)
    out = _orig_save(self, run_i, perf_dict, num_layers)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    return out


ResultManager.save_run_result = _save_with_mem

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

sys.argv = sys.argv[1:]
runpy.run_path(script, run_name="__main__")
"""

open(f"{SRC}/run_with_mem.py", "w").write(WRAPPER)
print("wrote", f"{SRC}/run_with_mem.py")

## 2. Smoke test the instrumentation

Five epochs, one run, into a throwaway exp-setting. Confirms the memory fields actually land in the JSON before you launch 120 runs.

In [ ]:
import subprocess, os, glob, json

ENV = {**os.environ, 'PYTHONUNBUFFERED': '1'}

r = subprocess.run([VENV_PY, 'run_with_mem.py', 'train_backprop.py',
    '--exp-setting','mem-smoke','--dataset','CitationFull-PubMed','--task','node-class',
    '--model','GNN-GAT','--num-layers','2','--num-runs','1','--seed','100',
    '--epochs','5','--val-every','2','--lr','0.001','--patience','5',
    '--num-hidden','128','--overwrite-result'],
    cwd=SRC, env=ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print('\n'.join(r.stdout.splitlines()[-15:]))
print('exit', r.returncode)

for f in glob.glob(f'{PROJECT_DIR}/results/mem-smoke/**/*.json', recursive=True):
    d = json.load(open(f))
    print({k: d[k] for k in d if 'mem' in k or k in ('perf','model')})

## 3. The job grid

All at `num_layers=4`, matching `nodeclass-sf.sh`.

`--overwrite-result` is omitted so an interrupted sweep resumes. Jobs run **dataset-major** so you get complete results for PubMed before GitHub starts — if you run out of time, you have whole datasets rather than fragments.

In [ ]:
import itertools, datetime, pathlib, time

DATASETS = ['CitationFull-PubMed', 'GitHub', 'Amazon-Photo']
ARCHS    = ['GAT', 'SAGE']
NUM_LAYERS = 4

METHODS = [
    ('train_backprop.py', 'GNN-{a}',                             []),
    ('train_forward.py',  'GNN_SingleForward-{a}',               []),
    ('train_forward.py',  'GNN_Top2Input_SingleForward-{a}',     ['--alternating-update']),
    ('train_forward.py',  'GNN_Top2Loss_SingleForward-{a}',      ['--alternating-update']),
]

FORWARD_EXTRA = ['--append-label', 'none', '--aug-edge-direction', 'bidirection']

def common(dataset):
    return ['--exp-setting','perf-results','--dataset',dataset,'--task','node-class',
            '--num-runs','5','--seed','100','--epochs','1000','--val-every','2',
            '--lr','0.001','--patience','100','--num-hidden','128',
            '--num-layers',str(NUM_LAYERS)]

JOBS = []
for ds, arch in itertools.product(DATASETS, ARCHS):
    for script, tmpl, extra in METHODS:
        model = tmpl.format(a=arch)
        args = common(ds) + ['--model', model] + list(extra)
        if script == 'train_forward.py':
            args += FORWARD_EXTRA
        JOBS.append({'script': script, 'model': model, 'dataset': ds, 'args': args})

print(f"{len(JOBS)} configs x 5 runs = {len(JOBS)*5} runs")
for j in JOBS[:4]: print(' ', j['dataset'], j['model'])

## 4. Runner

In [ ]:
LOG_DIR = pathlib.Path(PROJECT_DIR) / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def run_job(job, dry=False):
    cmd = [VENV_PY, 'run_with_mem.py', job['script']] + job['args']
    tag = f"{job['dataset']}-{job['model']}-L{NUM_LAYERS}"
    if dry:
        print(' '.join(cmd)); return 0
    log = LOG_DIR / f"{tag}-{datetime.datetime.now():%Y%m%d_%H%M%S}.log"
    print(f"\n{'='*78}\n  {tag}\n{'='*78}")
    t0 = time.time()
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, cwd=SRC, env=ENV, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            f.write(line)
            if any(k in line for k in ('Test Performance','Test Accuracy','Error',
                                       'Traceback','Starting run','Skipping run',
                                       'out of memory')):
                print(line.rstrip())
        p.wait()
    status = 'OK' if p.returncode == 0 else f'FAILED ({p.returncode})'
    print(f"  -> {status} in {time.time()-t0:.0f}s   log: {log.name}")
    return p.returncode

failed = []
print("Dry run:")
for j in JOBS: run_job(j, dry=True)

## 5. Run — one dataset at a time

Run these cells separately rather than all 24 at once. That way a failure on GitHub doesn't cost you the PubMed results, and you can check memory numbers look sane before committing more compute.

In [ ]:
failed = []
for j in [x for x in JOBS if x['dataset'] == 'CitationFull-PubMed']:
    if run_job(j) != 0: failed.append(j)
print('\nfailed:', [f"{x['dataset']}/{x['model']}" for x in failed])

In [ ]:
for j in [x for x in JOBS if x['dataset'] == 'GitHub']:
    if run_job(j) != 0: failed.append(j)
print('\nfailed:', [f"{x['dataset']}/{x['model']}" for x in failed])

In [ ]:
for j in [x for x in JOBS if x['dataset'] == 'Amazon-Photo']:
    if run_job(j) != 0: failed.append(j)
print('\nfailed:', [f"{x['dataset']}/{x['model']}" for x in failed])

## 6. Collect accuracy + memory

Note the scale normalisation: the backprop path writes accuracy as a fraction (`utils/eval_utils.py`), the forward paths write a percentage (`gnn_sf.py`). Verify this against your own JSONs before trusting the assert.

In [ ]:
import json, glob, pandas as pd, numpy as np

ARCH_OF = lambda m: m.rsplit('-', 1)[-1]
def method_of(m):
    if m.startswith('GNN-'):                        return 'BP'
    if m.startswith('GNN_SingleForward'):           return 'SF'
    if m.startswith('GNN_Top2Input_SingleForward'): return 'SF-Top2Input'
    if m.startswith('GNN_Top2Loss_SingleForward'):  return 'SF-Top2Loss'
    return m

rows = []
for f in glob.glob(f'{PROJECT_DIR}/results/perf-results/**/*.json', recursive=True):
    d = json.load(open(f))
    perf = d['perf'] * 100 if d['model'].startswith('GNN-') else d['perf']
    rows.append({'dataset': d['dataset'], 'arch': ARCH_OF(d['model']),
                 'method': method_of(d['model']), 'layers': d['num_layers'],
                 'run': d['run_i'], 'acc': perf,
                 'time': d.get('train_time'),
                 'mem': d.get('gpu_mem_alloc_peak_mb'),
                 'mem_res': d.get('gpu_mem_reserved_peak_mb')})

df = pd.DataFrame(rows)
assert not df.empty, 'No results found.'
assert df['acc'].between(0, 100).all(), 'Scale check failed - inspect raw perf values.'
print(f"{df['mem'].notna().sum()}/{len(df)} runs have memory recorded "
      f"(runs from before the wrapper was installed will be missing it)")

agg = (df.groupby(['dataset','arch','method','layers'])
         .agg(acc_mean=('acc','mean'), acc_std=('acc','std'),
              mem_mb=('mem','mean'), time_s=('time','mean'), n=('run','count'))
         .reset_index())
pd.set_option('display.width', 200)
print(agg.to_string(index=False, float_format=lambda v: f'{v:.2f}'))
agg.to_csv(f'{PROJECT_DIR}/summary_gat_sage.csv', index=False)

## 7. Memory comparison figure

The paper's memory claim is that forward learning avoids storing the backward graph. This is the plot that tests it: peak allocated memory per method, grouped by dataset and architecture, with BP as the reference bar.

In [ ]:
import matplotlib.pyplot as plt

m = agg.dropna(subset=['mem_mb'])
if m.empty:
    print('No memory data yet.')
else:
    order = ['BP','SF','SF-Top2Input','SF-Top2Loss']
    archs, dsets = sorted(m['arch'].unique()), sorted(m['dataset'].unique())
    fig, axes = plt.subplots(len(archs), 2, figsize=(13, 4.2*len(archs)), squeeze=False)
    for r, arch in enumerate(archs):
        for c, (metric, ylab) in enumerate([('mem_mb','Peak GPU memory (MB)'),
                                            ('acc_mean','Test accuracy (%)')]):
            ax = axes[r][c]
            w = 0.8/len(order)
            for i, meth in enumerate(order):
                sub = m[(m['arch']==arch) & (m['method']==meth)].set_index('dataset')
                vals = [sub[metric].get(d, np.nan) for d in dsets]
                ax.bar(np.arange(len(dsets)) + i*w - 0.4, vals, w, label=meth)
            ax.set_xticks(range(len(dsets)))
            ax.set_xticklabels([d.replace('CitationFull-','') for d in dsets])
            ax.set_ylabel(ylab); ax.set_title(f'{arch} - {ylab.split("(")[0].strip()}')
            ax.grid(alpha=0.3, axis='y')
            if r==0 and c==0: ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(f'{PROJECT_DIR}/memory_accuracy_gat_sage.png', dpi=150)
    plt.show()

    piv = m.pivot_table(index=['dataset','arch'], columns='method', values='mem_mb')
    if 'BP' in piv:
        for c in [x for x in piv.columns if x != 'BP']:
            piv[f'{c} vs BP'] = piv[c] / piv['BP']
    print(piv.to_string(float_format=lambda v: f'{v:.2f}'))